# AgriShield

End-to-end soil-risk pipeline: harmonise surveys, train a classifier, query live satellite + climate for a new site.

```
LUCAS + WoSIS  ->  data/agrishield_training.csv  ->  RandomForest
New lat/lon    ->  GEE (Sentinel-2, WorldClim, static soil)  ->  risk report
```

**Hard rule this notebook follows:** every column in `FEATURE_COLUMNS` (agrishield/config.py) must be obtainable live, from lat/lon alone, with no lab test. Lab-only chemistry (OC, N, P, K, EC, CEC, bulk density) is the TARGET, never a feature. `dataset.py` and `model.py` are unchanged from before; `config.py`, `gee.py`, `climate.py`, `inference.py` were rewritten to enforce this.

Run cells top to bottom. Section 2b (satellite/climate enrichment) is the slow one -- smoke test before the full run.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agrishield.config import TRAINING_CSV, FEATURE_COLUMNS, EXAMPLE_SITE
from agrishield.dataset import build_training_csv
from agrishield.climate import enrich_training_csv
from agrishield.model import train, save_model, predict_proba
from agrishield.inference import live_features, live_model_inputs

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
print("root:", ROOT)
print("training table exists:", TRAINING_CSV.exists())

## 2. Base training table -- LUCAS + WoSIS only

No APIs yet, pure local files. Fast (seconds, not minutes).

`sample_year` is kept as a lookup key for step 2b -- it decides which satellite image to fetch for each row. It is never a model feature; check `FEATURE_COLUMNS` above if you want to confirm.

In [ ]:
df = build_training_csv()
print("rows:", len(df), " cols:", df.shape[1])
print(df["source"].value_counts())
display(df.head())

### 2a. Know your date coverage before spending GEE quota

Sentinel-2 (the satellite you're querying in 2b) only exists from 2015 onward. Rows with an older or missing `sample_year` will end up with satellite columns as `NaN` -- expected, not a bug. WorldClim and static soil/elevation don't depend on date, so they'll still fill in for almost every row.

In [ ]:
has_year = df["sample_year"].notna()
print("missing sample_year:", df["sample_year"].isna().sum(),
      f"({df['sample_year'].isna().mean()*100:.1f}%)")
print("year < 2015 (pre-Sentinel-2):", (df.loc[has_year, "sample_year"] < 2015).sum())
print("year >= 2015 (real spectral data possible):", (df.loc[has_year, "sample_year"] >= 2015).sum())

## 2b. Attach satellite + climate + soil columns (slow -- do the smoke test first)

Three things get attached to the SAME rows as new columns:
- **Sentinel-2 bands + NDVI** -- batched per `sample_year` (one composite image per year, not one API call per row). Only fills for rows with `sample_year >= 2015`.
- **Static soil texture + elevation** -- from OpenLandMap/SRTM. Fills for virtually every row with coordinates, and OVERWRITES the LUCAS lab-measured `clay_pct`/`sand_pct`/`silt_pct`/`elevation_m` so training and live inference read from the identical source.
- **WorldClim** -- climatology, fills for virtually every row.

Run the 800-row smoke test first. Check the fill rates make sense. Only then uncomment the full run.

In [ ]:
# SMOKE TEST -- ~800 rows, should take a couple of minutes, not hours
_smoke = enrich_training_csv(batch_size=400, max_rows=None)
check_cols = ["B2", "B3", "B4", "ndvi", "clay_pct", "sand_pct", "elevation_m", "tmean_c", "precip_mm"]
print(_smoke[check_cols].notna().mean().round(3))

In [ ]:
# FULL RUN -- once the smoke test above looks sane. This is the "overnight" one for 232k rows.
# Safe to re-run if interrupted: already-filled rows are skipped (see climate.py _run_batches).
# df = enrich_training_csv(batch_size=400).
# df.to_csv(TRAINING_CSV, index=False)

## 3. Train

Loads from disk (not the in-memory `df` above) so this cell works even if you restarted the kernel after the overnight run finished.

In [ ]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
print("training on", len(df), "rows")
model, report = train(df)
print(report)
save_model(model)

import numpy as np
cols = [c for c in FEATURE_COLUMNS if c in df.columns]
importances = pd.Series(model.named_steps["clf"].feature_importances_, index=cols).sort_values(ascending=False)
print("\nfeature importances:")
print(importances)

### 3b. Optional comparison: satellite-era rows only (sample_year >= 2015)

You chose to train on everything first and check accuracy -- good, evidence beats guessing. This cell runs the same model on just the ~36k rows that have REAL (not imputed) spectral bands, so you can see whether accuracy actually improves on cleaner data or whether the extra 196k rows were pulling their weight.

In [ ]:
df_recent = df[df["sample_year"] >= 2015].copy()
print("rows with real satellite era data:", len(df_recent))
model_recent, report_recent = train(df_recent)
print("--- full 232k ---")
print(report)
print("--- satellite-era only (~36k) ---")
print(report_recent)

## 4. Live inference for a new site

This is what actually runs when a farmer taps Calculate -- one coordinate, live GEE + weather calls, no training data touched. `live_model_inputs` feeds the model; `live_features` wraps that plus current weather for a human-facing report.

In [ ]:
site = EXAMPLE_SITE
report_data = live_features(site["latitude"], site["longitude"])
print(site["name"], site["latitude"], site["longitude"])
report_data

In [ ]:
inputs = live_model_inputs(site["latitude"], site["longitude"])
print(inputs)
predict_proba(model, inputs)

In [ ]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
check_cols = ["B2","B3","B4","ndvi","clay_pct","sand_pct","silt_pct","elevation_m","tmean_c","precip_mm"]
print(df[check_cols].notna().mean().round(3))
print(df.groupby(df["sample_year"] >= 2015)[check_cols].apply(lambda x: x.notna().mean()))

In [ ]:
# check what actually got filled
check_cols = ["B2", "B3", "B4", "ndvi", "clay_pct", "sand_pct", "elevation_m", "tmean_c", "precip_mm"]
print(_smoke[check_cols].notna().mean().round(3))

# train — reload from disk to be safe, since this ran for ~2 hours and you may have restarted things
import pandas as pd
from agrishield.config import TRAINING_CSV
from agrishield.model import train, save_model

df = pd.read_csv(TRAINING_CSV, low_memory=False)
model, report = train(df)
print(report)
save_model(model)